# Flight Delays · цена ошибки и порог решения (Spark, 5.8 млн рейсов)

**Данные:** [`usdot/flight-delays`](https://www.kaggle.com/datasets/usdot/flight-delays) — все внутренние рейсы США за 2015 год
(Bureau of Transportation Statistics): ~5.8 млн строк, ~565 МБ CSV. Реальная статистика перевозчиков, не синтетика.

**Задача:** до вылета предсказать, прибудет ли рейс с опозданием ≥ 15 минут (определение BTS).
Признаки — только то, что известно **за `PRED_HORIZON_H` часов до планового вылета** (по умолчанию 2 ч): расписание, перевозчик, маршрут, загрузка аэропорта в тот же час
и — новое в v2 — **история**: прилёт предыдущего рейса того же борта (если он уже состоялся к моменту прогноза) и доля задержанных рейсов в аэропортах за последние 3 часа.

**Метрики карточки:** Sensitivity, Specificity, Precision, NPV, MCC, **AP** (Average Precision), **Brier Score**, F0.5 и F2.0.
Порог-зависимые метрики считаются по confusion matrix, AP и Brier — по скорам и от порога не зависят. Отдельно подбираются пороги,
максимизирующие F0.5 и F2 — и для положительного класса («задержка»), и для отрицательного («вовремя»).

**Тема карточки — цена ошибки.** Ложная тревога (FP) и пропущенная задержка (FN) стоят разных денег, поэтому «лучший порог» — не 0.5,
а точка минимума потерь. В отчёте сохраняем счётчики TP/FP/FN/TN для сетки порогов, а фронт пересчитывает стоимость под любые цены.

**Почему Spark:** 5.8 млн строк × десятки признаков + оконные агрегации загрузки аэропортов + три модели Spark ML.
Все тяжёлые операции идут на кластере (`spark-master` + `spark-worker`: 5 ядер / 8 ГБ executor), на драйвер возвращаются только гистограммы и сводки.
Если кластер не запущен — ноутбук сам переключится на `local[3]` и урежет данные (`DATA_FRACTION`).

**Что нужно в compose:** профиль `spark` (`docker compose --profile spark up -d`) и общий каталог, который видят и `ml`, и `spark-worker` (см. README к патчу).

### Что изменилось в v2 (по результатам разбора v1)

* **Потолок v1 — признаки, а не модель.** GBT / RandomForest / LogReg дали AP 0.278–0.294 и цену 15.6–15.8 тыс. $ на 1000 рейсов — разброс ≈1 %. Расписание, перевозчик и аэропорт почти не объясняют задержку; добавлена **история** (предыдущий рейс борта, ситуация в аэропортах) с жёстким горизонтом `PRED_HORIZON_H`, чтобы не было утечки.
* **Target-encoding out-of-fold** по дням (раньше строка train видела собственную метку), плюс кодирование маршрута и «аэропорт × час».
* **GBT с early stopping** по валидации, `maxBins=64`.
* **Bootstrap по календарным дням теста** → доверительные интервалы цены и попарное сравнение моделей (раньше «лучшая» определялась по разнице в 1 %, которая в пределах шума).
* **Prevalence в отчёте — по тесту** (19.0 %), а не по всей выборке (18.6 %): метрики теста сравниваются с baseline теста.
* Разбиение по дням не менялось (`hash(month, day) % 10`) → тестовая выборка та же, результаты сопоставимы с v1; **но задача другая** (прогноз за 2 ч до вылета, а не по расписанию), поэтому для честного A/B есть `USE_HISTORY=0`.

Переменные окружения: `PRED_HORIZON_H` (2), `USE_HISTORY` (1), `GBT_MAX_ITER` (150), `TREE_TRAIN_CAP`, `DATA_FRACTION`.

In [ ]:
import os, re, sys, json, math, time, socket, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / 'requirements.txt').exists() and (parent / 'notebooks').exists():
            return parent
    raise RuntimeError('Корень проекта не найден')


SEED    = 42
PROJECT = find_project_root()
TASK    = 'flight_delays'

MODEL_DIR = PROJECT / 'models' / 'classification' / TASK
ART_DIR   = PROJECT / 'artifacts' / 'classification' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT  :', PROJECT)
print('MODEL_DIR:', MODEL_DIR)
print('ART_DIR  :', ART_DIR)

## 1. Метрики без Spark (numpy)

Spark считает **одну гистограмму** скоров (1001 бин × 2 класса) на валидации и тесте. Из неё уже на драйвере, мгновенно и точно до 0.001,
получаем confusion matrix при любом пороге, кривые ROC/PR, лучший порог и все метрики. Это заменяет сотни повторных проходов по данным.
Функции проверены против `sklearn` (расхождение ~1e-16).

Дополнительно к той же гистограмме Spark считает **суммы score и score²** по бинам — из них получаем **точный Brier Score**
и диаграмму калибровки без повторных проходов. **AP** считается по гистограмме ступенчатой формулой `Σ (Rₙ − Rₙ₋₁)·Pₙ`
(как `sklearn.average_precision_score`, разрешение порога 0.001).

In [ ]:
# ---------- метрики без Spark: считаются по гистограммам / confusion matrix ----------
import math, json
import numpy as np

NBINS = 1000          # разрешение порога = 0.001


def _div(a, b):
    return a / b if b else 0.0


def _fbeta(p, r, beta):
    d = beta * beta * p + r
    return (1 + beta * beta) * p * r / d if d else 0.0


def binary_metrics(tp, fp, fn, tn):
    """Все метрики бинарной классификации из четырёх чисел confusion matrix."""
    tp, fp, fn, tn = (float(x) for x in (tp, fp, fn, tn))
    n, P, N = tp + fp + fn + tn, tp + fn, tn + fp
    prec_p, rec_p = _div(tp, tp + fp), _div(tp, P)
    prec_n, rec_n = _div(tn, tn + fn), _div(tn, N)
    f1p, f05p, f2p = (_fbeta(prec_p, rec_p, b) for b in (1.0, 0.5, 2.0))
    f1n, f05n, f2n = (_fbeta(prec_n, rec_n, b) for b in (1.0, 0.5, 2.0))
    acc = _div(tp + tn, n)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = _div(tp * tn - fp * fn, den)
    pe = _div((tp + fp) * P + (tn + fn) * N, n * n)
    kappa = _div(acc - pe, 1 - pe)
    return {
        'accuracy': acc,
        'balanced_accuracy': (rec_p + rec_n) / 2,
        'precision_macro': (prec_p + prec_n) / 2,
        'recall_macro': (rec_p + rec_n) / 2,
        'f1_macro': (f1p + f1n) / 2,
        'f1_weighted': _div(f1p * P + f1n * N, n),
        'f0.5_macro': (f05p + f05n) / 2,
        'f2_macro': (f2p + f2n) / 2,
        'mcc': mcc,
        'kappa': kappa,
        'specificity': rec_n,
        'sensitivity': rec_p,
        'precision_pos': prec_p,
        'recall_pos': rec_p,
        'f1_pos': f1p,
        'f0.5_pos': f05p,
        'f2_pos': f2p,
        # отрицательный класс («вовремя») как целевой: precision = NPV, recall = specificity
        'npv': prec_n,
        'precision_neg': prec_n,
        'recall_neg': rec_n,
        'f1_neg': f1n,
        'f0.5_neg': f05n,
        'f2_neg': f2n,
        'fpr': _div(fp, N),
    }


def hist_counts(neg, pos):
    """neg/pos — гистограммы длины NBINS+1 (бин = floor(score*NBINS)).
    Для порога k/NBINS (k = 0..NBINS) объект предсказан положительным при score >= k/NBINS.
    Возвращает массивы tp, fp, fn, tn длины NBINS+1."""
    neg = np.asarray(neg, dtype=np.float64)
    pos = np.asarray(pos, dtype=np.float64)
    neg_ge = np.cumsum(neg[::-1])[::-1]
    pos_ge = np.cumsum(pos[::-1])[::-1]
    P, N = pos.sum(), neg.sum()
    return pos_ge, neg_ge, P - pos_ge, N - neg_ge          # tp, fp, fn, tn


def _mcc_vec(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.divide(num, den, out=np.zeros_like(num, dtype=np.float64), where=den > 0)


F_KEYS = ('f0.5_pos', 'f2_pos', 'f0.5_neg', 'f2_neg')


def _f_vec(p, r, beta):
    b2 = beta * beta
    den = b2 * p + r
    return np.divide((1 + b2) * p * r, den, out=np.zeros_like(den, dtype=np.float64), where=den > 0)


def f_curves(neg, pos):
    """F0.5 и F2 для всех порогов k/NBINS — для положительного и отрицательного класса.
    Положительный: precision = TP/(TP+FP), recall = TP/P. Отрицательный: precision = NPV = TN/(TN+FN), recall = TN/N."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    z = lambda a, b: np.divide(a, b, out=np.zeros_like(a, dtype=np.float64), where=b > 0)
    prec_p, rec_p = z(tp, tp + fp), z(tp, tp + fn)
    prec_n, rec_n = z(tn, tn + fn), z(tn, tn + fp)
    return {'f0.5_pos': _f_vec(prec_p, rec_p, 0.5), 'f2_pos': _f_vec(prec_p, rec_p, 2.0),
            'f0.5_neg': _f_vec(prec_n, rec_n, 0.5), 'f2_neg': _f_vec(prec_n, rec_n, 2.0)}


def best_threshold_k(neg, pos, objective='mcc', cost_fp=1.0, cost_fn=1.0):
    """Индекс k лучшего порога на (валидационной) гистограмме.
    objective: 'mcc' (максимум) | 'cost' (минимум fp*cost_fp + fn*cost_fn) |
               'f0.5_pos' | 'f2_pos' | 'f0.5_neg' | 'f2_neg' (максимум F-меры нужного класса)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    if objective == 'mcc':
        return int(np.argmax(_mcc_vec(tp, fp, fn, tn)))
    if objective == 'cost':
        return int(np.argmin(fp * cost_fp + fn * cost_fn))
    if objective in F_KEYS:
        return int(np.argmax(f_curves(neg, pos)[objective]))
    raise ValueError(objective)


def val_best_block(neg_v, pos_v):
    """Лучшие пороги по F0.5/F2 (оба класса), подобранные на ВАЛИДАЦИИ."""
    fc = f_curves(neg_v, pos_v)
    out = {}
    for key in F_KEYS:
        k = int(np.argmax(fc[key]))
        out[key] = {'threshold': k / NBINS, 'val_score': float(fc[key][k])}
    return out


def average_precision(neg, pos):
    """Average Precision = Σ (R_k − R_{k+1}) · P_k по порогам k/NBINS от мягкого к жёсткому (как sklearn)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P = tp[0]
    if P <= 0:
        return 0.0
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    rec = tp / P
    rec_next = np.append(rec[1:], 0.0)                 # recall при более строгом пороге; за последним — 0
    return float(np.sum((rec - rec_next) * prec))


def brier_from_moments(neg, pos, mom):
    """Точный Brier Score = mean((score − y)²) из сумм по бинам: Σ(s²) − 2·Σ_{y=1}(s) + n_pos."""
    n = float(neg.sum() + pos.sum())
    num = mom['s2_neg'].sum() + mom['s2_pos'].sum() - 2.0 * mom['s1_pos'].sum() + pos.sum()
    return float(num / n) if n else float('nan')


def calibration_block(neg, pos, mom, n_bins=10):
    """Диаграмма калибровки: n_bins равных интервалов score → (средний score, доля задержек, размер)."""
    g = np.minimum((np.arange(NBINS + 1) * n_bins) // NBINS, n_bins - 1)
    n = np.bincount(g, weights=neg + pos, minlength=n_bins)
    n_pos = np.bincount(g, weights=pos, minlength=n_bins)
    s1 = np.bincount(g, weights=mom['s1_neg'] + mom['s1_pos'], minlength=n_bins)
    ok = n > 0
    mean_pred, obs = s1[ok] / n[ok], n_pos[ok] / n[ok]
    ece = float(np.sum(n[ok] / n.sum() * np.abs(obs - mean_pred)))
    return {'mean_pred': mean_pred, 'obs_rate': obs, 'n': n[ok], 'ece': ece}


def sweep_block(neg, pos, n_points=101):
    """Подвыборка порогов 0..1 для фронта: счётчики tp/fp/fn/tn на тестовой гистограмме."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    ks = np.unique(np.round(np.linspace(0, NBINS, n_points)).astype(int))
    return {
        'thresholds': [round(float(k) / NBINS, 4) for k in ks],
        'tp': [int(tp[k]) for k in ks], 'fp': [int(fp[k]) for k in ks],
        'fn': [int(fn[k]) for k in ks], 'tn': [int(tn[k]) for k in ks],
    }


def _thin(x, y):
    out_x, out_y = [], []
    for a, b in zip(x, y):
        if out_x and abs(a - out_x[-1]) < 2e-3 and abs(b - out_y[-1]) < 2e-3:
            continue
        out_x.append(round(float(a), 4)); out_y.append(round(float(b), 4))
    return out_x, out_y


def _cap(x, y, n=140):
    if len(x) <= n:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, n).round().astype(int))
    return [x[i] for i in idx], [y[i] for i in idx]


def curves_block(neg, pos):
    """ROC и PR-кривые по гистограмме (для графиков; точные AUC считает Spark)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P, N = max(tp[0], 1.0), max(fp[0], 1.0)
    tpr, fpr = tp / P, fp / N
    prec = np.divide(tp, tp + fp, out=np.ones_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    fx, fy = _cap(*_thin(fpr[::-1], tpr[::-1]))
    rx, ry = _cap(*_thin(tpr[::-1], prec[::-1]))
    return {'roc': {'fpr': fx, 'tpr': fy}, 'pr': {'recall': rx, 'precision': ry}}


def multiclass_metrics(cm):
    """cm[i][j] — сколько объектов истинного класса i предсказано как j."""
    cm = np.asarray(cm, dtype=np.float64)
    K, s = cm.shape[0], cm.sum()
    t, p, d = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    prec = np.divide(d, p, out=np.zeros(K), where=p > 0)
    rec = np.divide(d, t, out=np.zeros(K), where=t > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    acc = d.sum() / s
    mcc_den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0))
    mcc = (d.sum() * s - (p * t).sum()) / mcc_den if mcc_den else 0.0
    pe = (p * t).sum() / (s * s)
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    idx = np.arange(K)
    w = (idx[:, None] - idx[None, :]) ** 2 / max((K - 1) ** 2, 1)
    expected = np.outer(t, p) / s
    qwk = 1 - (w * cm).sum() / (w * expected).sum() if (w * expected).sum() else 0.0
    return {
        'accuracy': acc, 'balanced_accuracy': rec.mean(),
        'precision_macro': prec.mean(), 'recall_macro': rec.mean(), 'f1_macro': f1.mean(),
        'precision_weighted': (prec * t).sum() / s, 'recall_weighted': (rec * t).sum() / s,
        'f1_weighted': (f1 * t).sum() / s, 'mcc': mcc, 'kappa': kappa, 'qwk': qwk,
        '_per_class': {'precision': prec, 'recall': rec, 'f1': f1, 'support': t},
    }


def to_native(o):
    """Рекурсивно приводит numpy/NaN/Inf к JSON-совместимым значениям (NaN → null)."""
    if isinstance(o, dict):
        return {str(k): to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_native(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_native(o.tolist())
    if isinstance(o, np.generic):
        return to_native(o.item())
    if isinstance(o, float):
        return None if (math.isnan(o) or math.isinf(o)) else o
    if hasattr(o, '__fspath__'):
        return str(o)
    return o


def dump_json(obj, path):
    path.write_text(json.dumps(to_native(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

## 2. Spark: сессия, ресурсы, общий каталог

Бюджет берём из `docker-compose.yml`: executor — 10 ядер и 10 ГБ (worker ограничен 12 ГБ).
Python-UDF не используем вообще: только Spark SQL / Spark ML, поэтому версия Python у executor не критична и нет накладных расходов на сериализацию.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from pyspark import StorageLevel
import requests

# ---- бюджет ресурсов берём из docker-compose ----
# spark-worker:  command --cores 10 --memory 10g  (limit cpus=10, memory=12g)
# ml (driver):   лимит cpus=3.0, memory=6g       → драйвер 2g, локально не считаем
SPARK_MASTER_URL = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
CORES      = int(os.environ.get('SPARK_CORES', os.environ.get('WEATHER_SPARK_CORES', '10')))                # worker: --cores 10
EXEC_MEM   = os.environ.get('SPARK_EXECUTOR_MEMORY', os.environ.get('WEATHER_SPARK_EXECUTOR_MEMORY', '8g'))  # worker: --memory 10g, limit 12g
DRIVER_MEM = os.environ.get('SPARK_DRIVER_MEMORY', '2g')       # контейнер ml ограничен 6g — драйвер держим скромным
LOCAL_CORES = int(os.environ.get('SPARK_LOCAL_CORES', '3'))    # контейнер ml: cpus 3.0 (только для SPARK_MODE=local)
FORCE_MODE = os.environ.get('SPARK_MODE', 'cluster')           # cluster (по умолчанию) | auto | local
ALLOW_LOCAL_FALLBACK = os.environ.get('ALLOW_LOCAL_FALLBACK', '0') == '1'
EVENT_DIR  = Path('/opt/spark/events')                         # общий каталог для Spark History Server


def _master_tcp_ok(url, timeout=2.0):
    try:
        host, port = url.split('://', 1)[1].split(':')
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except Exception:
        return False


def _alive_workers(url):
    """Список живых worker из Web UI мастера (8085). None — UI недоступен."""
    try:
        host = url.split('://', 1)[1].split(':')[0]
        info = requests.get(f'http://{host}:8085/json/', timeout=3).json()
        return [w for w in info.get('workers', []) if w.get('state') == 'ALIVE']
    except Exception:
        return None


def choose_mode():
    """
    cluster (default) — требуем живой spark-master с ALIVE-worker. Нет → RuntimeError.
    auto             — пробуем cluster, при неудаче fallback на local (с громким предупреждением).
    local            — принудительно local[3] (для отладки; урезает данные, ест 3 ядра ml).
    ALLOW_LOCAL_FALLBACK=1 дополнительно разрешает fallback в режиме cluster.
    """
    if FORCE_MODE == 'local':
        print(f'⚠ SPARK_MODE=local: работаем на {LOCAL_CORES} ядрах ml (данные урежутся, только для отладки)')
        return 'local'

    if FORCE_MODE not in ('cluster', 'auto'):
        raise ValueError(f'SPARK_MODE={FORCE_MODE!r} (ожидалось auto|cluster|local)')

    if _master_tcp_ok(SPARK_MASTER_URL):
        workers = _alive_workers(SPARK_MASTER_URL)
        if workers is None or len(workers) > 0:
            return 'cluster'
        print('⚠ spark-master отвечает, но ALIVE-worker нет — задачи выполнять некому')

    if FORCE_MODE == 'auto' or ALLOW_LOCAL_FALLBACK:
        print(f'⚠ падаю в fallback local[{LOCAL_CORES}] — это НЕ полный прогон, 3 ядра ml, DATA_FRACTION=0.15')
        return 'local'

    raise RuntimeError(
        'Spark-кластер недоступен, а SPARK_MODE=cluster (по умолчанию).\n'
        f'  master: {SPARK_MASTER_URL}\n'
        '  Запустите профиль:\n'
        '      docker compose --profile spark up -d\n'
        '  Проверить: http://localhost:8085 (master UI), http://localhost:8086 (worker UI)\n'
        '  Для локальной отладки без кластера:\n'
        '      SPARK_MODE=local     (урежет данные до 15%, 3 ядра ml)\n'
        '      SPARK_MODE=auto      (попробует cluster, потом local)\n'
    )


def start_spark(mode):
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
    b = (SparkSession.builder.appName(f'ml-playground · {TASK}')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.driver.maxResultSize', '512m'))
    if mode == 'cluster':
        b = (b.master(SPARK_MASTER_URL)
              .config('spark.submit.deployMode', 'client')     # driver — здесь, executor — на worker
              .config('spark.driver.memory', DRIVER_MEM)
              .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))
              .config('spark.driver.bindAddress', '0.0.0.0')
              .config('spark.driver.port', '7078')
              .config('spark.blockManager.port', '7079')
              .config('spark.executor.memory', EXEC_MEM)       # 8g
              .config('spark.executor.cores', str(CORES))      # 10
              .config('spark.cores.max', str(CORES))           # 10 → 1 executor × 10 cores на worker
              .config('spark.sql.shuffle.partitions', str(CORES * 6))   # 60
              .config('spark.default.parallelism', str(CORES * 6)))     # 60
    else:
        b = (b.master(f'local[{LOCAL_CORES}]')
              .config('spark.driver.memory', '4g')
              .config('spark.sql.shuffle.partitions', str(LOCAL_CORES * 4)))
    if EVENT_DIR.is_dir():
        b = b.config('spark.eventLog.enabled', 'true').config('spark.eventLog.dir', f'file://{EVENT_DIR}')
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark


# ---------- ОБЩИЙ КАТАЛОГ (виден драйверу И executor-ам) ----------
# По docker-compose.yml в оба контейнера (ml и spark-worker) смонтированы:
#   ./data               → /opt/spark/data      (ml + worker)
#   ./models             → /app/models          (ml + worker)
#   ./spark-state/events → /opt/spark/events    (ml + worker)
# Плюс в ml есть `.:/app`, поэтому /app/data в ml — это ТОТ ЖЕ хост-каталог ./data,
# что и /opt/spark/data в ml и worker. Именно поэтому копировать ./data/flight-delays
# в ./data/spark_cache/raw/ незачем: worker уже видит его по /opt/spark/data/flight-delays.
#
# Имя проверочного файла — probe.txt, БЕЗ префикса '_'.
# Hadoop FileInputFormat молча игнорирует файлы с '_' или '.' в начале имени
# (то же правило прячет _SUCCESS и .crc в parquet), поэтому
#   spark.read.text('file:///.../_probe.txt').collect() == []
# и раньше это давало ложное «executor не видят каталог».
SHARED_CANDIDATES = [
    Path('/opt/spark/data/spark_cache'),       # ./data               → ml + worker
    Path('/app/models/.spark_cache'),          # ./models             → ml + worker
    Path('/opt/spark/events/spark_cache'),     # ./spark-state/events → ml + worker
    PROJECT / 'data' / 'spark_cache',          # fallback для local[3] / нестандартного compose
]


def pick_shared_dir(spark, mode):
    if mode == 'local':
        d = PROJECT / 'data' / 'spark_cache'
        d.mkdir(parents=True, exist_ok=True)
        print(f'✓ shared dir: {d} — local[3], executor в том же процессе')
        return d
    for d in SHARED_CANDIDATES:
        try:
            d.mkdir(parents=True, exist_ok=True)
            probe = d / 'probe.txt'                # НЕ _probe.txt (см. комментарий выше)
            probe.write_text('ok')
            got = spark.read.text(f'file://{probe}').collect()   # читает executor, не драйвер
            if got and got[0][0] == 'ok':
                print(f'✓ shared dir: {d} — виден executor-ам')
                return d
            print(f'  {d}: probe прочитан пустым (got={got!r})')
        except Exception as e:
            print(f'  {d}: executor не видят каталог ({type(e).__name__})')
    return None


MODE  = choose_mode()
spark = start_spark(MODE)
SHARED = pick_shared_dir(spark, MODE)

# fallback в local разрешён только через choose_mode (по SPARK_MODE/ALLOW_LOCAL_FALLBACK)
# если сюда попали с MODE=cluster и SHARED=None — это проблема монтирования, а не режима
if SHARED is None:
    raise RuntimeError(
        'Ни один общий каталог не виден executor-ам. Проверьте docker-compose:\n'
        '  spark-worker должен монтировать ./data, ./models или ./spark-state/events\n'
        '  по тому же пути, что и ml (см. SHARED_CANDIDATES).\n'
        '  Кандидаты: ' + ', '.join(str(p) for p in SHARED_CANDIDATES)
    )

sc = spark.sparkContext
DATA_FRACTION = float(os.environ.get('DATA_FRACTION', '1.0' if MODE == 'cluster' else '0.15'))

SPARK_INFO = {
    'mode': MODE,
    'master': SPARK_MASTER_URL if MODE == 'cluster' else f'local[{LOCAL_CORES}]',
    'spark_version': spark.version,
    'app_id': sc.applicationId,
    'cores': CORES if MODE == 'cluster' else LOCAL_CORES,
    'executor_memory': EXEC_MEM if MODE == 'cluster' else '4g (driver)',
    'default_parallelism': int(sc.defaultParallelism),
    'shuffle_partitions': int(spark.conf.get('spark.sql.shuffle.partitions')),
    'shared_dir': str(SHARED),
    'data_fraction': DATA_FRACTION,
}
print(json.dumps(SPARK_INFO, indent=2, ensure_ascii=False))
if MODE == 'local':
    print(f'\n⚠ local-режим: данные урезаны до {DATA_FRACTION:.0%} (DATA_FRACTION), используются {LOCAL_CORES} ядра ml. '
          'Для полного объёма на 10 ядрах spark-worker: docker compose --profile spark up -d и SPARK_MODE=cluster.')


# ---------- утилиты ----------
TIMINGS = {}


class stage:
    """with stage('имя'): ...  — печатает и копит время этапов (идёт в report.json)."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f'▶ {self.name} ...')
        return self

    def __exit__(self, *exc):
        TIMINGS[self.name] = round(time.time() - self.t, 1)
        print(f'✔ {self.name}: {TIMINGS[self.name]} с')


def ensure_shared_dir(src):
    """
    Приводит src к пути, видимому executor-ам (под SHARED / /opt/spark/data).
    Копирует только когда действительно надо.

    Ключевой нюанс: в ml есть ДВА монтирования на один и тот же хост-каталог ./data —
       .:/app               → /app/data       (= ./data на хосте)
       ./data:/opt/spark/data → /opt/spark/data (= ./data на хосте)
    Поэтому /app/data/flight-delays и /opt/spark/data/flight-delays — ОДИН И ТОТ ЖЕ файл
    на хосте. spark-worker монтирует только ./data:/opt/spark/data, значит видит его как
    /opt/spark/data/flight-delays. Копирование в SHARED/raw/ тут не нужно и только плодит
    дубли; а попытка copytree на несуществующем пути — это та самая FileNotFoundError.
    """
    src = Path(src)
    try:
        src = src.resolve()
    except OSError:
        pass
    shared = SHARED.resolve()

    # 1) Уже под SHARED — отлично, ничего не делаем
    try:
        src.relative_to(shared)
        if src.exists():
            return src
    except ValueError:
        pass

    # 2) /app/data/... и /opt/spark/data/... — тот же хост-каталог ./data, что и у worker-а.
    #    Пересчитываем на /opt/spark/data/<rel> без копирования (работает и для файлов, и для каталогов).
    for alias in (Path('/app/data'), Path('/opt/spark/data')):
        try:
            rel = src.relative_to(alias.resolve())
        except ValueError:
            continue
        worker_path = Path('/opt/spark/data') / rel
        if worker_path.exists():
            print(f'✓ {src} → {worker_path} (тот же хост-каталог ./data, копия не нужна)')
            return worker_path

    # 3) Если src вообще не существует — внятная ошибка вместо падения copytree
    if not src.exists():
        raise FileNotFoundError(
            f'Нет данных: {src}\n'
            f'  kagglehub недоступен и локальной копии нет.\n'
            f'  Положите файлы на хост, например:\n'
            f'    ./data/{src.name}/    (в контейнере это /opt/spark/data/{src.name}/)\n'
            f'  или настройте доступ к Kaggle (KAGGLE_USERNAME / KAGGLE_KEY).\n'
        )

    # 4) Обычный кейс — копируем под SHARED/raw/<name>
    dst = shared / 'raw' / src.name
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() and dst.resolve() == src:
            return src
        print(f'копирую файл {src} → {dst} (чтобы executor увидели файл)')
        shutil.copy2(src, dst)
        return dst
    if dst.exists() and dst.resolve() == src:
        return src
    print(f'копирую {src} → {dst} (чтобы executor увидели файлы)')
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return dst


def spark_path(p):
    return f'file://{p}'

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator


def score_sdf(model, sdf, extra=()):
    """Предсказания модели → DataFrame(label, *extra, score = P(класс 1))."""
    return model.transform(sdf).select('label', *extra, vector_to_array('probability')[1].alias('score'))


def score_histogram(sdf, moments=False):
    """Гистограмма score по NBINS+1 бинам отдельно для классов 0 и 1 — один проход по данным.
    Из неё без повторных заданий получаем confusion matrix при ЛЮБОМ пороге k/NBINS.
    moments=True дополнительно возвращает суммы score и score² по бинам (для точного Brier Score и калибровки)."""
    b = F.greatest(F.lit(0), F.least(F.floor(F.col('score') * NBINS).cast('int'), F.lit(NBINS)))
    g = (sdf.select(b.alias('b'), F.col('label').cast('int').alias('y'), F.col('score').alias('s'))
            .groupBy('b', 'y')
            .agg(F.count('*').alias('c'), F.sum('s').alias('s1'), F.sum(F.col('s') * F.col('s')).alias('s2')))
    rows = g.collect()
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    mom = {k: np.zeros(NBINS + 1) for k in ('s1_neg', 's1_pos', 's2_neg', 's2_pos')}
    for r in rows:
        side = 'pos' if r['y'] == 1 else 'neg'
        (pos if side == 'pos' else neg)[r['b']] = r['c']
        mom[f's1_{side}'][r['b']] = r['s1']
        mom[f's2_{side}'][r['b']] = r['s2']
    return (neg, pos, mom) if moments else (neg, pos)


def spark_auc(sdf):
    """Точные ROC-AUC и PR-AUC (без даунсэмплинга кривых) средствами Spark."""
    out = {}
    for metric, key in (('areaUnderROC', 'roc_auc'), ('areaUnderPR', 'pr_auc')):
        try:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric, numBins=0)
        except TypeError:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric)
        out[key] = float(ev.evaluate(sdf))
    return out


def tree_importance(model, names):
    imp = model.featureImportances.toArray()
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])

## 3. Загрузка и очистка данных

`kagglehub` скачивает датасет сразу в общий каталог (`KAGGLEHUB_CACHE`), поэтому executor читают CSV напрямую — без копирования через драйвер.

Очистка: оставляем выполненные рейсы (без `CANCELLED`/`DIVERTED`), цель — `ARRIVAL_DELAY >= 15`.

**Признаки загрузки аэропорта** (`origin_cong`, `dest_cong`) — сколько рейсов запланировано из/в этот аэропорт в тот же час того же дня.
Это агрегат по расписанию (без меток), его знают заранее; считается группировкой по всем 5.8 млн строк.

**Разбиение по календарным дням.** Целые сутки уходят в один набор (hash от месяца и дня → 70% train / 10% val / 20% test).
Случайное построчное разбиение дало бы утечку: рейсы одного дня делят погоду и загрузку, и модель «узнавала бы» день.

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = str(SHARED / 'kagglehub')
import kagglehub

with stage('Загрузка usdot/flight-delays'):
    try:
        DATA_DIR = Path(kagglehub.dataset_download('usdot/flight-delays'))
    except Exception as e:
        print('kagglehub недоступен:', e)
        DATA_DIR = PROJECT / 'data' / 'flight-delays'
        print('использую локальную копию:', DATA_DIR)

DATA_DIR = ensure_shared_dir(DATA_DIR)
flights_csv, airlines_csv = DATA_DIR / 'flights.csv', DATA_DIR / 'airlines.csv'
assert flights_csv.exists(), f'нет файла {flights_csv}'
print(f'flights.csv: {flights_csv.stat().st_size / 1e6:.0f} МБ')

raw = spark.read.option('header', True).csv(spark_path(flights_csv))
print('столбцов:', len(raw.columns))

In [ ]:
from pyspark.sql import Window
from functools import reduce

# ---- горизонт прогноза и признаки истории ----
PRED_HORIZON_H = int(os.environ.get('PRED_HORIZON_H', '2'))   # прогноз делается за столько часов до планового вылета
USE_HISTORY    = os.environ.get('USE_HISTORY', '1') == '1'     # 0 → только статические признаки (как в v1) — для A/B
RECENT_HOURS   = 3                                            # окно «недавней ситуации» в аэропорту, часов
assert PRED_HORIZON_H >= 1, 'горизонт меньше часа не поддерживается: история считается по полным часам'
H_SEC = PRED_HORIZON_H * 3600

sel = raw.select(
    F.col('YEAR').cast('int').alias('year'),
    F.col('MONTH').cast('int').alias('month'),
    F.col('DAY').cast('int').alias('day'),
    F.col('DAY_OF_WEEK').cast('int').alias('dow'),
    F.col('AIRLINE').alias('airline'),
    F.col('TAIL_NUMBER').alias('tail'),
    F.col('ORIGIN_AIRPORT').alias('origin'),
    F.col('DESTINATION_AIRPORT').alias('dest'),
    F.col('SCHEDULED_DEPARTURE').cast('int').alias('sched_dep'),
    F.col('SCHEDULED_ARRIVAL').cast('int').alias('sched_arr'),
    F.col('SCHEDULED_TIME').cast('double').alias('sched_time'),
    F.col('DISTANCE').cast('double').alias('distance'),
    F.col('DEPARTURE_DELAY').cast('double').alias('dep_delay'),   # ТОЛЬКО как история ДРУГИХ рейсов; у самого рейса не используется
    F.col('ARRIVAL_DELAY').cast('double').alias('arr_delay'),
    F.col('CANCELLED').cast('int').alias('cancelled'),
    F.col('DIVERTED').cast('int').alias('diverted'),
)

with stage('Подсчёт строк в исходном CSV'):
    N_RAW = sel.count()
print(f'строк в CSV: {N_RAW:,}')

two_h = lambda c: (F.floor(F.col(c) / 100).cast('int') % 24)
flown = (sel
    .where((F.col('cancelled') == 0) & (F.col('diverted') == 0)
           & F.col('arr_delay').isNotNull() & F.col('dep_delay').isNotNull() & F.col('sched_time').isNotNull()
           & F.col('distance').isNotNull() & F.col('sched_dep').isNotNull() & F.col('sched_arr').isNotNull())
    .withColumn('label', (F.col('arr_delay') >= 15).cast('double'))
    .withColumn('dep_hour', two_h('sched_dep'))
    .withColumn('arr_hour', two_h('sched_arr'))
    .withColumn('dep_tod', (F.col('dep_hour') * 60 + (F.col('sched_dep') % 100)).cast('double'))
    .withColumn('is_weekend', (F.col('dow') >= 6).cast('double'))
    .withColumn('fold_id', F.pmod(F.xxhash64('month', 'day'), F.lit(10)))      # то же разбиение, что в v1 → тест сравним
    .withColumn('fold', F.when(F.col('fold_id') < 7, 'train').when(F.col('fold_id') == 7, 'val').otherwise('test'))
    # ---- локальные «наивные» времена в секундах (часовой пояс сессии = UTC; внутри одного аэропорта часы согласованы) ----
    .withColumn('tail_key', F.coalesce(F.col('tail'), F.concat(F.lit('NA_'), F.monotonically_increasing_id().cast('string'))))
    .withColumn('day_ts', F.make_date('year', 'month', 'day').cast('timestamp').cast('long'))
    .withColumn('sched_dep_ts', F.col('day_ts') + two_h('sched_dep') * 3600 + (F.col('sched_dep') % 100) * 60)
    .withColumn('sched_arr_ts', F.col('day_ts') + two_h('sched_arr') * 3600 + (F.col('sched_arr') % 100) * 60
                + F.when(F.col('sched_arr') < F.col('sched_dep'), 86400).otherwise(0))
    .withColumn('act_dep_ts', (F.col('sched_dep_ts') + F.col('dep_delay') * 60).cast('long'))
    .withColumn('act_arr_ts', (F.col('sched_arr_ts') + F.col('arr_delay') * 60).cast('long')))

# ---- (1) предыдущий рейс того же борта: известен ли его прилёт к моменту прогноза t = плановый вылет − H ----
w_tail = Window.partitionBy('tail_key').orderBy('sched_dep_ts', 'sched_arr_ts')
flown = (flown
    .withColumn('p_act_arr', F.lag('act_arr_ts').over(w_tail))
    .withColumn('p_sch_arr', F.lag('sched_arr_ts').over(w_tail))
    .withColumn('p_arr_delay', F.lag('arr_delay').over(w_tail))
    .withColumn('p_dest', F.lag('dest').over(w_tail)))
has_prev = F.col('p_act_arr').isNotNull() & (F.col('p_dest') == F.col('origin'))
t_pred = F.col('sched_dep_ts') - F.lit(H_SEC)
flown = (flown
    # запас на оборот по расписанию, минуты (всегда известен из расписания)
    .withColumn('prev_slack', F.when(has_prev, F.least(F.greatest((F.col('sched_dep_ts') - F.col('p_sch_arr')) / 60.0, F.lit(-60.0)),
                                                       F.lit(1440.0))).otherwise(F.lit(1440.0)))
    # прилёт предыдущего рейса состоялся ДО момента прогноза → его задержка известна
    .withColumn('prev_known', F.when(has_prev & (F.col('p_act_arr') <= t_pred), 1.0).otherwise(0.0))
    # на сколько предыдущий рейс «залез» за плановый вылет этого (если известно; иначе 0 + prev_known=0)
    .withColumn('prev_overrun', F.when(F.col('prev_known') == 1.0,
                                       F.least(F.greatest(F.col('p_arr_delay') - F.col('prev_slack'), F.lit(-120.0)), F.lit(300.0))
                                       ).otherwise(0.0)))
flown_base = flown.persist(StorageLevel.MEMORY_AND_DISK)


# ---- (2) недавняя ситуация в аэропортах: рейсы, которые ФАКТИЧЕСКИ вылетели/прилетели за RECENT_HOURS полных часов до t ----
def rolling_table(src, key, ts_col, delay_col, prefix):
    """Скользящая сумма по RECENT_HOURS часам: строка (k, hr) = последние RECENT_HOURS часов, заканчивающиеся часом hr включительно.
    Реализовано сдвигом почасовой таблицы (hr+0, hr+1, ...) и суммой — без оконных функций и без «дыр» в часах без рейсов."""
    base = src.select(F.col(key).alias('k'), F.floor(F.col(ts_col) / 3600).cast('long').alias('hr'), F.col(delay_col).alias('d'))
    hourly = base.groupBy('k', 'hr').agg(
        F.count('*').alias('n'),
        F.sum((F.col('d') >= 15).cast('double')).alias('nd'),
        F.sum(F.least(F.greatest(F.col('d'), F.lit(-15.0)), F.lit(180.0))).alias('sd'))
    shifted = reduce(lambda a, b: a.unionByName(b), [hourly.withColumn('hr', F.col('hr') + s) for s in range(RECENT_HOURS)])
    roll = shifted.groupBy('k', 'hr').agg(F.sum('n').alias('n'), F.sum('nd').alias('nd'), F.sum('sd').alias('sd'))
    return roll.select(F.col('k').alias(f'{prefix}_k'), F.col('hr').alias(f'{prefix}_hr'), F.col('n').alias(f'{prefix}_n'),
                       (F.col('nd') / F.col('n')).alias(f'{prefix}_rate'), (F.col('sd') / F.col('n')).alias(f'{prefix}_mean'))


o_roll = rolling_table(flown_base, 'origin', 'act_dep_ts', 'dep_delay', 'o_rec')
d_roll = rolling_table(flown_base, 'dest', 'act_arr_ts', 'arr_delay', 'd_rec')
# последний ПОЛНЫЙ час до t: всё, что в нём вылетело/прилетело, уже случилось к моменту прогноза.
# для аэропорта назначения t переводится в его локальные часы: t_dest = плановый прилёт − плановое время в пути − H
flown = (flown_base
    .withColumn('o_ref_hr', F.floor((F.col('sched_dep_ts') - F.lit(H_SEC)) / 3600).cast('long') - 1)
    .withColumn('d_ref_hr', F.floor((F.col('sched_arr_ts') - F.col('sched_time') * 60 - F.lit(H_SEC)) / 3600).cast('long') - 1)
    .join(o_roll, (F.col('origin') == F.col('o_rec_k')) & (F.col('o_ref_hr') == F.col('o_rec_hr')), 'left')
    .join(d_roll, (F.col('dest') == F.col('d_rec_k')) & (F.col('d_ref_hr') == F.col('d_rec_hr')), 'left'))

# Загрузка аэропортов по расписанию (без меток): рейсов в тот же час того же дня
dep_cong = flown_base.groupBy('origin', 'month', 'day', 'dep_hour').agg(F.count('*').alias('origin_cong'))
arr_cong = flown_base.groupBy('dest', 'month', 'day', 'arr_hour').agg(F.count('*').alias('dest_cong'))
flown = (flown.join(dep_cong, ['origin', 'month', 'day', 'dep_hour'], 'left')
              .join(arr_cong, ['dest', 'month', 'day', 'arr_hour'], 'left'))

KEEP = ['label', 'fold', 'fold_id', 'month', 'day', 'dow', 'airline', 'origin', 'dest', 'dep_hour', 'arr_hour', 'dep_tod',
        'is_weekend', 'sched_time', 'distance', 'origin_cong', 'dest_cong',
        'prev_slack', 'prev_known', 'prev_overrun', 'o_rec_n', 'o_rec_rate', 'o_rec_mean', 'd_rec_n', 'd_rec_rate', 'd_rec_mean']
flown = flown.select(*KEEP)

if DATA_FRACTION < 1.0:                       # только в local-режиме; признаки уже посчитаны на полном объёме
    flown = flown.sample(False, DATA_FRACTION, SEED)

CLEAN_PATH = SHARED / 'flights_clean.parquet'
with stage('Очистка + загрузка аэропортов + запись parquet'):
    flown.write.mode('overwrite').parquet(spark_path(CLEAN_PATH))
flown_base.unpersist()

df = spark.read.parquet(spark_path(CLEAN_PATH))
N_USED = df.count()
print(f'после очистки: {N_USED:,} рейсов ({N_USED / N_RAW:.1%} от CSV) · партиций: {df.rdd.getNumPartitions()}')

# ---- диагностика качества данных и покрытия истории ----
cov = df.agg(F.avg('prev_known').alias('prev_known'), F.avg((F.col('o_rec_n') > 0).cast('double')).alias('o_cov'),
             F.avg((F.col('d_rec_n') > 0).cast('double')).alias('d_cov'),
             F.avg(F.col('origin').rlike('^[0-9]+$').cast('double')).alias('num_origin')).first()
print(f'горизонт прогноза: {PRED_HORIZON_H} ч · история: {"ВКЛ" if USE_HISTORY else "ВЫКЛ"}')
print(f'прилёт предыдущего рейса борта известен к моменту прогноза: {cov["prev_known"]:.1%}')
print(f'есть недавние рейсы в аэропорту вылета: {cov["o_cov"]:.1%} · прилёта: {cov["d_cov"]:.1%}')
print(f'ORIGIN_AIRPORT — числовой код вместо IATA: {cov["num_origin"]:.2%}  (если заметно > 0 — один аэропорт делится на два ключа)')

## 4. EDA на Spark

Агрегаты считаются кластером, на драйвер приходят десятки строк. Те же числа уйдут во фронт (`report.eda`).

In [ ]:
airlines_pd = pd.read_csv(airlines_csv).rename(columns={'IATA_CODE': 'code', 'AIRLINE': 'name'})
AIRLINE_NAMES = dict(zip(airlines_pd.code, airlines_pd.name))


def rate_by(col):
    rows = (df.groupBy(col).agg(F.count('*').alias('n'), F.avg('label').alias('rate')).orderBy(col).collect())
    return [{col: r[col], 'n': int(r['n']), 'rate': float(r['rate'])} for r in rows]


with stage('EDA-агрегаты'):
    eda_hour, eda_month, eda_dow = rate_by('dep_hour'), rate_by('month'), rate_by('dow')
    eda_airline = [{'code': r['airline'], 'name': AIRLINE_NAMES.get(r['airline'], r['airline']), 'n': r['n'], 'rate': r['rate']}
                   for r in rate_by('airline')]
    eda_airline.sort(key=lambda d: -d['rate'])
    fold_rows = df.groupBy('fold').agg(F.count('*').alias('n'), F.avg('label').alias('rate')).collect()

FOLDS = {r['fold']: {'n': int(r['n']), 'prevalence': float(r['rate'])} for r in fold_rows}
PREVALENCE = sum(FOLDS[f]['n'] * FOLDS[f]['prevalence'] for f in FOLDS) / N_USED
print('доля задержек ≥15 мин:', f'{PREVALENCE:.3%}')
for f, v in FOLDS.items():
    print(f'  {f:5s} n={v["n"]:>9,}  prevalence={v["prevalence"]:.3%}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
h = pd.DataFrame(eda_hour);  axes[0].bar(h.dep_hour, h.rate * 100, color='#6366f1'); axes[0].set_title('Доля задержек по часу вылета'); axes[0].set_xlabel('час'); axes[0].set_ylabel('%')
a = pd.DataFrame(eda_airline); axes[1].barh(a.code, a.rate * 100, color='#8b5cf6'); axes[1].set_title('По авиакомпаниям'); axes[1].invert_yaxis(); axes[1].set_xlabel('%')
m = pd.DataFrame(eda_month); axes[2].bar(m.month, m.rate * 100, color='#0ea5e9'); axes[2].set_title('По месяцам'); axes[2].set_xlabel('месяц')
plt.tight_layout(); plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight'); plt.show()

## 5. Признаки

* **Target-encoding** (`origin_rate`, `dest_rate`, `route_rate`, `origin_hour_rate`, `dest_hour_rate`): сглаженная доля задержек (`m=100`, сглаживание к prior), считается **только по train**; для самих строк train — **out-of-fold по дням**, чтобы метка строки не попадала в её же признак.
* **История (v2)**, всё строго до момента прогноза `t = плановый вылет − H`:
  * `prev_known`, `prev_slack`, `prev_overrun` — прилетел ли предыдущий рейс борта до `t`, запас на оборот по расписанию, на сколько предыдущий рейс «залез» за плановый вылет;
  * `o_rec_*` / `d_rec_*` — число рейсов, доля задержек ≥15 мин и средняя задержка среди рейсов, **фактически** вылетевших из / прилетевших в аэропорт за 3 полных часа до `t`.
* `airline` — категориальный индекс (деревья) и one-hot (логистическая регрессия).
* Для линейной модели добавлены циклические `sin/cos` часа и месяца, `log(distance)`, срезанные версии `prev_*`.
* `DEPARTURE_DELAY` и `ARRIVAL_DELAY` самого рейса, `TAXI_*`, `*_DELAY` в признаки **не** попадают — это утечка цели.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.stat import Summarizer

train_raw = df.where(F.col('fold') == 'train')
PRIOR = float(train_raw.agg(F.avg('label')).first()[0])
TE_M = 100.0


def add_te(frame, cols, out):
    """Сглаженный target-encoding по ключу cols, считается ТОЛЬКО по train.
    Для строк train — out-of-fold по дням (свой fold_id исключается), для val/test — по всему train.
    Так модель не видит собственную метку в признаке, и распределение признака на train ≈ как на test."""
    cols = list(cols)
    per_fold = train_raw.groupBy(*cols, 'fold_id').agg(F.sum('label').alias('_s_f'), F.count('*').alias('_n_f'))
    total = per_fold.groupBy(*cols).agg(F.sum('_s_f').alias('_s_t'), F.sum('_n_f').alias('_n_t'))
    f = frame.join(F.broadcast(total), cols, 'left').join(F.broadcast(per_fold), cols + ['fold_id'], 'left')
    s = F.coalesce(F.col('_s_t'), F.lit(0.0)) - F.coalesce(F.col('_s_f'), F.lit(0.0))
    n = F.coalesce(F.col('_n_t'), F.lit(0)) - F.coalesce(F.col('_n_f'), F.lit(0))
    return f.withColumn(out, (s + F.lit(TE_M * PRIOR)) / (n + F.lit(TE_M))).drop('_s_t', '_n_t', '_s_f', '_n_f')


TE_SPECS = [(['origin'], 'origin_rate'), (['dest'], 'dest_rate'), (['origin', 'dest'], 'route_rate'),
            (['origin', 'dep_hour'], 'origin_hour_rate'), (['dest', 'arr_hour'], 'dest_hour_rate')]

with stage('Признаки: target-encoding, индексы, векторы'):
    feat = df
    for cols_, out_ in TE_SPECS:
        feat = add_te(feat, cols_, out_)
    feat = feat.fillna({'origin_cong': 1, 'dest_cong': 1, 'o_rec_n': 0, 'd_rec_n': 0,
                        'o_rec_rate': PRIOR, 'd_rec_rate': PRIOR, 'o_rec_mean': 0.0, 'd_rec_mean': 0.0})

    two_pi = 2 * math.pi
    feat = (feat.withColumn('hour_sin', F.sin(F.col('dep_hour') * two_pi / 24))
                .withColumn('hour_cos', F.cos(F.col('dep_hour') * two_pi / 24))
                .withColumn('month_sin', F.sin(F.col('month') * two_pi / 12))
                .withColumn('month_cos', F.cos(F.col('month') * two_pi / 12))
                .withColumn('log_distance', F.log1p(F.col('distance')))
                .withColumn('prev_overrun_pos', F.greatest(F.col('prev_overrun'), F.lit(0.0)))   # для линейной модели
                .withColumn('prev_slack_c', F.least(F.col('prev_slack'), F.lit(240.0))))

    idx_model = StringIndexer(inputCol='airline', outputCol='airline_idx', handleInvalid='keep').fit(feat.where(F.col('fold') == 'train'))
    feat = idx_model.transform(feat)
    ohe_model = OneHotEncoder(inputCols=['airline_idx'], outputCols=['airline_ohe'], handleInvalid='keep').fit(feat)
    feat = ohe_model.transform(feat)

    STATIC_TREE = ['month', 'dow', 'is_weekend', 'dep_hour', 'dep_tod', 'arr_hour', 'distance', 'sched_time', 'airline_idx',
                   'origin_rate', 'dest_rate', 'route_rate', 'origin_hour_rate', 'dest_hour_rate', 'origin_cong', 'dest_cong']
    HIST_TREE = ['prev_slack', 'prev_known', 'prev_overrun', 'o_rec_n', 'o_rec_rate', 'o_rec_mean', 'd_rec_n', 'd_rec_rate', 'd_rec_mean']
    STATIC_LR = ['is_weekend', 'dow', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'log_distance', 'sched_time',
                 'origin_rate', 'dest_rate', 'route_rate', 'origin_hour_rate', 'dest_hour_rate', 'origin_cong', 'dest_cong']
    HIST_LR = ['prev_known', 'prev_slack_c', 'prev_overrun_pos', 'o_rec_n', 'o_rec_rate', 'o_rec_mean', 'd_rec_n', 'd_rec_rate', 'd_rec_mean']
    TREE_FEATURES = STATIC_TREE + (HIST_TREE if USE_HISTORY else [])
    LR_NUMERIC = STATIC_LR + (HIST_LR if USE_HISTORY else [])
    AIRLINES = list(idx_model.labelsArray[0])
    LR_NAMES = LR_NUMERIC + [f'airline={a}' for a in AIRLINES]

    feat = VectorAssembler(inputCols=TREE_FEATURES, outputCol='features').transform(feat)
    feat = VectorAssembler(inputCols=LR_NUMERIC + ['airline_ohe'], outputCol='features_lr').transform(feat)
    feat = feat.select('label', 'fold', 'month', 'day', 'features', 'features_lr')

    train = feat.where(F.col('fold') == 'train').persist(StorageLevel.MEMORY_AND_DISK)
    val   = feat.where(F.col('fold') == 'val').persist(StorageLevel.MEMORY_AND_DISK)
    test  = feat.where(F.col('fold') == 'test').persist(StorageLevel.MEMORY_AND_DISK)
    N_TRAIN, N_VAL, N_TEST = train.count(), val.count(), test.count()

print(f'train={N_TRAIN:,}  val={N_VAL:,}  test={N_TEST:,}')
print('признаков (деревья):', len(TREE_FEATURES), '| (LogReg):', len(LR_NAMES))

## 6. Модели и «цена ошибки»

Цены ошибок — **допущение**, их можно менять во фронте:

| Ошибка | Что значит | Цена по умолчанию |
|---|---|---|
| **FP** | предупредили о задержке, рейс пришёл вовремя (лишние переносы, потеря доверия к уведомлениям) | 25 $ |
| **FN** | пропустили задержку (сорванная стыковка, компенсации) | 100 $ |

Порог каждой модели подбирается **на валидации** по минимуму суммарной цены, а метрики и цена считаются **на тесте**.
Для калиброванной модели оптимум ≈ `C_FP / (C_FP + C_FN)` = 0.2 — это проверка на здравый смысл.

Кроме цены считаются **Sensitivity, Specificity, Precision, NPV, MCC, AP, Brier Score, F0.5 и F2.0**.
Пороги, максимизирующие F0.5 и F2 (и для «задержки», и для «вовремя»), подбираются на валидации и сохраняются в `val_best`.

Два baseline'а без модели — «всегда вовремя» и «всегда задержка». Любая полезная модель обязана быть дешевле обоих.
Деревья учатся на выборке train не более `TREE_TRAIN_CAP` строк — ограничение памяти executor (8 ГБ); логистическая регрессия видит весь train.

GBT обучается с `validationIndicatorCol`: валидационные дни входят в DataFrame с флагом, и Spark останавливает бустинг, когда logloss на val перестаёт улучшаться. После обучения считаются **доверительные интервалы цены** (bootstrap по дням теста) — «лучшая модель» объявляется, но в отчёт попадает и то, различимы ли модели статистически.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier

COST_FP, COST_FN, COST_UNIT = 25.0, 100.0, '$'
TREE_TRAIN_CAP = int(os.environ.get('TREE_TRAIN_CAP', 3_000_000))


def make_entry(name, family, vh, th, aucs, importance=None, train_s=None, fixed_k=None, is_baseline=False, mom=None):
    """Одна модель → запись для report.json. vh/th = (neg, pos) гистограммы на val/test,
    mom — суммы score и score² по бинам теста (для Brier Score и калибровки)."""
    neg_v, pos_v = vh
    neg_t, pos_t = th
    k = fixed_k if fixed_k is not None else best_threshold_k(neg_v, pos_v, 'cost', COST_FP, COST_FN)
    tp, fp, fn, tn = hist_counts(neg_t, pos_t)
    n_t = float(tp[0] + fp[0])
    prev_t_ = float(tp[0] / n_t)
    m = binary_metrics(tp[k], fp[k], fn[k], tn[k])
    m.update(aucs)
    ap = average_precision(neg_t, pos_t)
    brier = brier_from_moments(neg_t, pos_t, mom) if mom is not None else None
    ref = prev_t_ * (1 - prev_t_)
    m.update({'ap': ap, 'brier': brier, 'brier_skill': (1 - brier / ref) if (brier is not None and ref > 0) else None})
    k5 = NBINS // 2
    vb = val_best_block(neg_v, pos_v)
    at_f = {}                                          # метрики на тесте при порогах, выбранных на валидации по F0.5 / F2
    for key, v in vb.items():
        kk = int(round(v['threshold'] * NBINS))
        at_f[key] = {'threshold': v['threshold'], **binary_metrics(tp[kk], fp[kk], fn[kk], tn[kk])}
    return {
        'name': name, 'model': family, 'is_baseline': bool(is_baseline), 'is_best': False,
        'threshold': k / NBINS,
        'metrics': m,
        'roc_auc': aucs['roc_auc'], 'pr_auc': aucs['pr_auc'],
        'ap': ap, 'brier': brier,
        'calibration': calibration_block(neg_t, pos_t, mom) if mom is not None else None,
        'val_best': vb, 'at_f_thresholds': at_f,
        'confusion_counts': {'tp': int(tp[k]), 'fp': int(fp[k]), 'fn': int(fn[k]), 'tn': int(tn[k])},
        'cost_per_1000': float((fp[k] * COST_FP + fn[k] * COST_FN) / n_t * 1000),
        'fixed_05': binary_metrics(tp[k5], fp[k5], fn[k5], tn[k5]),
        'sweep': sweep_block(neg_t, pos_t),
        'curves': curves_block(neg_t, pos_t),
        'feature_importance': importance or [],
        'train_seconds': train_s,
    }


def baseline_hist(const_score, like):
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    b = 0 if const_score == 0 else NBINS
    neg[b], pos[b] = like[0].sum(), like[1].sum()
    return neg, pos


def baseline_moments(const_score, th):
    """Суммы score и score² для константного предсказания (нужны для точного Brier baseline)."""
    neg, pos = th
    mom = {k: np.zeros(NBINS + 1) for k in ('s1_neg', 's1_pos', 's2_neg', 's2_pos')}
    if const_score == 1:
        b = NBINS
        mom['s1_neg'][b] = mom['s2_neg'][b] = neg.sum()
        mom['s1_pos'][b] = mom['s2_pos'][b] = pos.sum()
    return mom


def lr_importance(model, names, std):
    imp = np.abs(model.coefficients.toArray()) * std
    imp = imp / imp.sum() if imp.sum() else imp
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])


frac_tree = min(1.0, TREE_TRAIN_CAP / N_TRAIN)
tree_train = train.sample(False, frac_tree, SEED).persist(StorageLevel.MEMORY_AND_DISK) if frac_tree < 1.0 else train
LR_STD = Summarizer.metrics('std').summary(F.col('features_lr')).alias('s')
LR_STD = np.array(train.select(LR_STD).first()['s']['std'].toArray())
print(f'деревья учатся на {frac_tree:.0%} train (≈{int(N_TRAIN * frac_tree):,} строк)')

GBT_MAX_ITER = int(os.environ.get('GBT_MAX_ITER', 150))     # верхняя граница; реально остановится раньше (early stopping по val)

# GBT: val идёт в обучающий DataFrame с флагом is_val → Spark сам останавливает бустинг по logloss на val (validationIndicatorCol).
gbt_fit = (tree_train.withColumn('is_val', F.lit(False))
           .unionByName(val.withColumn('is_val', F.lit(True)))).persist(StorageLevel.MEMORY_AND_DISK)

SPECS = [
    ('LogReg', 'LogisticRegression', train,
     LogisticRegression(featuresCol='features_lr', labelCol='label', maxIter=60, regParam=1e-4, elasticNetParam=0.0)),
    ('RandomForest', 'RandomForest', tree_train,
     RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=60, maxDepth=12, maxBins=32,
                            subsamplingRate=0.3, minInstancesPerNode=20, featureSubsetStrategy='onethird', seed=SEED)),
    ('GBT', 'GradientBoosting', gbt_fit,
     GBTClassifier(featuresCol='features', labelCol='label', maxIter=GBT_MAX_ITER, maxDepth=6, stepSize=0.15,
                   subsamplingRate=0.5, maxBins=64, minInstancesPerNode=100,
                   validationIndicatorCol='is_val', validationTol=0.001, seed=SEED)),
]


def day_counts(sdf, k):
    """tp/fp/fn/tn по каждому календарному дню теста при пороге k/NBINS (тот же критерий бина, что в score_histogram)."""
    pred = F.floor(F.col('score') * NBINS) >= F.lit(k)
    y = F.col('label') == 1.0
    agg = sdf.groupBy('month', 'day').agg(
        F.sum((pred & y).cast('long')).alias('tp'), F.sum((pred & ~y).cast('long')).alias('fp'),
        F.sum((~pred & y).cast('long')).alias('fn'), F.sum((~pred & ~y).cast('long')).alias('tn'))
    rows = sorted(agg.collect(), key=lambda r: (r['month'], r['day']))
    keys = [(int(r['month']), int(r['day'])) for r in rows]
    cnt = np.array([[r['tp'], r['fp'], r['fn'], r['tn']] for r in rows], dtype=np.float64)
    return keys, cnt


def day_bootstrap(tables, B=2000, level=0.95):
    """Bootstrap по КАЛЕНДАРНЫМ ДНЯМ теста (рейсы одного дня зависимы: общая погода и сбои) → доверительные интервалы цены.
    Один и тот же набор дней для всех моделей → можно честно сравнивать модели попарно."""
    names = list(tables)
    keys = tables[names[0]][0]
    assert all(tables[nm][0] == keys for nm in names), 'наборы дней у моделей различаются'
    M = np.stack([tables[nm][1] for nm in names])                  # (модели, дни, [tp, fp, fn, tn])
    D = M.shape[1]
    idx = np.random.default_rng(SEED).integers(0, D, size=(B, D))
    tot = M[:, idx, :].sum(axis=2)                                 # (модели, B, 4)
    n_fl = tot.sum(axis=2)
    cost = (tot[..., 1] * COST_FP + tot[..., 2] * COST_FN) / n_fl * 1000
    pos, neg = tot[0][:, 0] + tot[0][:, 2], tot[0][:, 1] + tot[0][:, 3]           # метки одни и те же у всех моделей
    base = {'Baseline · всегда вовремя': pos * COST_FN / n_fl[0] * 1000, 'Baseline · всегда задержка': neg * COST_FP / n_fl[0] * 1000}
    lo, hi = (1 - level) / 2 * 100, (1 + level) / 2 * 100
    ci = lambda a: {'lo': float(np.percentile(a, lo)), 'hi': float(np.percentile(a, hi)), 'level': level, 'n_days': int(D), 'boots': B}
    return {'names': names, 'matrix': cost, 'base': base,
            'cost': {**{nm: ci(cost[i]) for i, nm in enumerate(names)}, **{b: ci(v) for b, v in base.items()}}}


entries, fitted, hists, day_tables = [], {}, {}, {}
for name, family, tr, est in SPECS:
    with stage(f'обучение · {name}') as st:
        model = est.fit(tr)
    train_s = TIMINGS[f'обучение · {name}']
    if family == 'GradientBoosting':
        print(f'   GBT: построено деревьев {model.getNumTrees} из {GBT_MAX_ITER} (early stopping по val)')
    with stage(f'оценка · {name}'):
        val_s = score_sdf(model, val).persist()
        test_s = score_sdf(model, test, extra=('month', 'day')).persist()
        vh = score_histogram(val_s)                                  # (neg, pos)
        neg_t, pos_t, mom_t = score_histogram(test_s, moments=True)  # (neg, pos, mom)
        th = (neg_t, pos_t)                                          # make_entry ждёт кортеж
        aucs = spark_auc(test_s)
        imp = lr_importance(model, LR_NAMES, LR_STD) if family == 'LogisticRegression' else tree_importance(model, TREE_FEATURES)
        entries.append(make_entry(name, family, vh, th, aucs, imp, train_s, mom=mom_t))
        day_tables[name] = day_counts(test_s, int(round(entries[-1]['threshold'] * NBINS)))
        val_s.unpersist(); test_s.unpersist()
    fitted[name], hists[name] = model, (vh, th)
    e = entries[-1]
    print(f'   {name}: thr={e["threshold"]:.3f}  cost/1000={e["cost_per_1000"]:,.0f}{COST_UNIT}  AP={e["ap"]:.4f}  Brier={e["brier"]:.4f}  PR-AUC={e["pr_auc"]:.4f}  ROC-AUC={e["roc_auc"]:.4f}')
gbt_fit.unpersist()

# --- baseline'ы без модели ---
ref_hist = next(iter(hists.values()))
prev_t = float(ref_hist[1][1].sum() / (ref_hist[1][0].sum() + ref_hist[1][1].sum()))
for bname, const in (('Baseline · всегда вовремя', 0), ('Baseline · всегда задержка', 1)):
    vh, th = baseline_hist(const, ref_hist[0]), baseline_hist(const, ref_hist[1])
    entries.append(make_entry(bname, 'Baseline', vh, th, {'roc_auc': 0.5, 'pr_auc': prev_t},
                              fixed_k=NBINS // 2, is_baseline=True, mom=baseline_moments(const, th)))

models_only = [e for e in entries if not e['is_baseline']]
best = min(models_only, key=lambda e: e['cost_per_1000'])
best['is_best'] = True
print('\nЛучшая по цене:', best['name'], f'{best["cost_per_1000"]:,.0f}{COST_UNIT} на 1000 рейсов')

# --- доверительные интервалы цены (bootstrap по дням теста) и попарное сравнение с лучшей моделью ---
BOOT = day_bootstrap(day_tables)
for e in entries:
    e['cost_ci'] = BOOT['cost'][e['name']]
bi = BOOT['names'].index(best['name'])
base_best = np.minimum(BOOT['base']['Baseline · всегда вовремя'], BOOT['base']['Baseline · всегда задержка'])
sav = 1 - BOOT['matrix'][bi] / base_best
PAIRED = {}
for i, nm in enumerate(BOOT['names']):
    if nm == best['name']:
        continue
    d = BOOT['matrix'][i] - BOOT['matrix'][bi]                    # > 0 → лучшая модель дешевле
    PAIRED[nm] = {'p_best_cheaper': float((d > 0).mean()), 'diff_mean': float(d.mean()),
                  'diff_lo': float(np.percentile(d, 2.5)), 'diff_hi': float(np.percentile(d, 97.5))}
SAVING_CI = {'mean': float(sav.mean()), 'lo': float(np.percentile(sav, 2.5)), 'hi': float(np.percentile(sav, 97.5))}
BOOT_REPORT = {'method': 'bootstrap по календарным дням теста', 'n_days': BOOT['cost'][best['name']]['n_days'],
               'boots': BOOT['cost'][best['name']]['boots'], 'prices': {'fp': COST_FP, 'fn': COST_FN},
               'paired_vs_best': PAIRED, 'saving_vs_best_baseline': SAVING_CI}

print(f'\nЦена на 1000 рейсов, 95% ДИ (bootstrap по {BOOT_REPORT["n_days"]} дням теста, цены FP={COST_FP:.0f}, FN={COST_FN:.0f}):')
for e in entries:
    c = e['cost_ci']
    print(f'   {e["name"]:<28s} {e["cost_per_1000"]:>8,.0f}   [{c["lo"]:>8,.0f} … {c["hi"]:>8,.0f}]')
for nm, p in PAIRED.items():
    print(f'   {best["name"]} дешевле {nm} в {p["p_best_cheaper"]:.0%} бутстрэп-выборок · разница {p["diff_mean"]:,.0f}{COST_UNIT} [{p["diff_lo"]:,.0f} … {p["diff_hi"]:,.0f}]')
print(f'   экономия лучшей модели против лучшего baseline: {SAVING_CI["mean"]:.1%}  [{SAVING_CI["lo"]:.1%} … {SAVING_CI["hi"]:.1%}]')

## 7. Сводка

Читаем одновременно **цену**, **AP**, **Brier Score** и **MCC**: модель может выигрывать по ROC-AUC, а по деньгам — нет, если стоимости ошибок несимметричны.
AP (ступенчатая PR-площадь) заменяет ROC-кривую: она считается только по положительному классу и не раздувается числом верно отпущенных рейсов.
Brier Score оценивает сами вероятности — отдельно от порога (ниже — лучше, ориентир `prev·(1−prev)`).

In [ ]:
tbl = pd.DataFrame([{
    'model': e['name'], 'thr': round(e['threshold'], 3), f'cost/1000 {COST_UNIT}': round(e['cost_per_1000']),
    'FP': e['confusion_counts']['fp'], 'FN': e['confusion_counts']['fn'],
    'Sensitivity': round(e['metrics']['sensitivity'], 4), 'Specificity': round(e['metrics']['specificity'], 4),
    'Precision': round(e['metrics']['precision_pos'], 4), 'NPV': round(e['metrics']['npv'], 4),
    'MCC': round(e['metrics']['mcc'], 4), 'AP': round(e['ap'], 4), 'Brier': round(e['brier'], 4),
    'F0.5': round(e['metrics']['f0.5_pos'], 4), 'F2.0': round(e['metrics']['f2_pos'], 4),
    'ROC-AUC': round(e['roc_auc'], 4),
    'cost 95% CI': f"{e['cost_ci']['lo']:,.0f}–{e['cost_ci']['hi']:,.0f}",
} for e in entries]).set_index('model')
display(tbl)
tbl.to_csv(ART_DIR / 'metrics_all_models.csv')


def pr_steps(e):
    """Ступенчатая PR-кривая (precision справа от шага — как в формуле AP)."""
    return e['curves']['pr']['recall'], e['curves']['pr']['precision']


# ---- 1. цена ошибок + PR-кривые со ступенчатой интерполяцией (AP в легенде) ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for e in entries:
    sw = e['sweep']
    n_t = sw['tp'][0] + sw['fp'][0]
    cost = [(f * COST_FP + n * COST_FN) / n_t * 1000 for f, n in zip(sw['fp'], sw['fn'])]
    axes[0].plot(sw['thresholds'], cost, '--' if e['is_baseline'] else '-', lw=2, label=e['name'])
    if not e['is_baseline']:
        r, p = pr_steps(e)
        axes[1].step(r, p, where='pre', lw=2, label=f'{e["name"]} · AP={e["ap"]:.3f}')
axes[0].set_xlabel('порог'); axes[0].set_ylabel(f'потери на 1000 рейсов, {COST_UNIT}'); axes[0].set_title(f'Цена ошибок (FP={COST_FP:.0f}{COST_UNIT}, FN={COST_FN:.0f}{COST_UNIT})'); axes[0].legend(fontsize=8)
axes[1].axhline(prev_t, color='k', ls='--', alpha=.5, label=f'prevalence={prev_t:.3f}'); axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision'); axes[1].set_title('Precision–Recall (ступенчатая) и AP, тест'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'cost_and_pr.png', bbox_inches='tight'); plt.show()

# ---- 2. AP, Brier Score, калибровка ----
trained = [e for e in entries if not e['is_baseline']]
names = [e['name'].replace('Baseline · ', 'base · ') for e in entries]
colors = ['#94a3b8' if e['is_baseline'] else '#6366f1' for e in entries]
best_e = next(e for e in entries if e['is_best'])
brier_ref = prev_t * (1 - prev_t)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
ax = axes[0]
bars = ax.bar(names, [e['ap'] for e in entries], color=colors)
ax.axhline(prev_t, color='k', ls='--', alpha=.5, label=f'без модели = prevalence {prev_t:.3f}')
for b_, e in zip(bars, entries):
    ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f'{e["ap"]:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Average Precision (выше — лучше)'); ax.set_ylabel('AP'); ax.legend(fontsize=8); ax.tick_params(axis='x', rotation=20)

ax = axes[1]
bars = ax.bar(names, [e['brier'] for e in entries], color=colors)
ax.axhline(brier_ref, color='k', ls='--', alpha=.5, label=f'ориентир prev·(1−prev) = {brier_ref:.4f}')
for b_, e in zip(bars, entries):
    ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f'{e["brier"]:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Brier Score (ниже — лучше)'); ax.set_ylabel('Brier'); ax.legend(fontsize=8); ax.tick_params(axis='x', rotation=20)

ax = axes[2]
ax.plot([0, 1], [0, 1], 'k--', alpha=.5, label='идеальная калибровка')
for e in trained:
    c = e['calibration']
    ax.plot(c['mean_pred'], c['obs_rate'], 'o-', lw=2.5 if e['is_best'] else 1.5, label=f'{e["name"]} · ECE={c["ece"]:.3f}')
ax.set_xlabel('предсказанная вероятность'); ax.set_ylabel('фактическая доля задержек'); ax.set_title('Калибровка (10 интервалов, тест)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'ap_brier.png', bbox_inches='tight'); plt.show()

### Порог по F0.5 и F2 — для положительного и отрицательного сценария

* **Положительный сценарий** — целевой класс «задержка»: precision = TP/(TP+FP), recall = Sensitivity.
* **Отрицательный сценарий** — целевой класс «вовремя»: precision = **NPV** = TN/(TN+FN), recall = **Specificity**.

F0.5 делает упор на precision (меньше ложных тревог), F2 — на recall (меньше пропусков). Пороги подбираются на **валидации** (`val_best`),
метрики при них считаются на тесте (`at_f_thresholds`). Во фронте у каждого из четырёх оптимумов есть кнопка «выбрать лучший».

Ниже — confusion matrix **в стандартном виде**: строки — фактический класс, столбцы — предсказанный, положительный класс первым
(`[[TP, FN], [FP, TN]]`). Внимание: у `sklearn.metrics.confusion_matrix` порядок обратный — `[[TN, FP], [FN, TP]]`.

In [ ]:
# ---- пороги по F0.5 / F2 для лучшей модели ----
fc_test = f_curves(*hists[best_e['name']][1])
ks = np.arange(NBINS + 1) / NBINS
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, side, title in ((axes[0], 'pos', 'Положительный сценарий: «задержка»'), (axes[1], 'neg', 'Отрицательный сценарий: «вовремя»')):
    for beta, col in (('f0.5', '#0ea5e9'), ('f2', '#f59e0b')):
        key = f'{beta}_{side}'
        ax.plot(ks, fc_test[key], color=col, lw=2, label=beta.upper())
        kv = int(round(best_e['val_best'][key]['threshold'] * NBINS))          # порог, выбранный на валидации
        kt = int(np.argmax(fc_test[key]))                                      # максимум на тесте (для сравнения)
        ax.scatter([ks[kv]], [fc_test[key][kv]], color=col, s=70, zorder=5, label=f'{beta} · порог val = {ks[kv]:.3f}')
        ax.scatter([ks[kt]], [fc_test[key][kt]], color=col, s=70, zorder=5, marker='x', label=f'{beta} · максимум на тесте = {ks[kt]:.3f}')
    ax.axvline(best_e['threshold'], color='#6366f1', ls='--', alpha=.7, label=f'порог по цене = {best_e["threshold"]:.3f}')
    ax.set_title(f'{title} — {best_e["name"]}'); ax.set_xlabel('порог'); ax.set_ylabel('F-мера'); ax.set_ylim(0, 1.02); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'fbeta_threshold.png', bbox_inches='tight'); plt.show()

rows_f = []
for key, v in best_e['at_f_thresholds'].items():
    rows_f.append({'оптимум': key, 'порог (val)': round(v['threshold'], 3), 'Sensitivity': round(v['sensitivity'], 4),
                   'Specificity': round(v['specificity'], 4), 'Precision': round(v['precision_pos'], 4), 'NPV': round(v['npv'], 4),
                   'MCC': round(v['mcc'], 4), 'F0.5': round(v['f0.5_pos'], 4), 'F2.0': round(v['f2_pos'], 4)})
display(pd.DataFrame(rows_f).set_index('оптимум'))

# ---- confusion matrix: стандартный вид [[TP, FN], [FP, TN]] ----
cc_ = best_e['confusion_counts']
cm_std = np.array([[cc_['tp'], cc_['fn']], [cc_['fp'], cc_['tn']]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, data, fmt, ttl in ((axes[0], cm_std, ',d', 'Количество'), (axes[1], cm_std / cm_std.sum(axis=1, keepdims=True), '.1%', 'Доля от строки')):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', cbar=False, ax=ax, annot_kws={'size': 14},
                xticklabels=['Positive (задержка)', 'Negative (вовремя)'], yticklabels=['Positive (задержка)', 'Negative (вовремя)'])
    ax.set_xlabel('Predicted (предсказано)'); ax.set_ylabel('Actual (факт)'); ax.set_title(f'{ttl} · {best_e["name"]}, порог {best_e["threshold"]:.3f}')
plt.tight_layout(); plt.savefig(ART_DIR / 'confusion_matrix.png', bbox_inches='tight'); plt.show()
print(f'Sensitivity={cc_["tp"] / (cc_["tp"] + cc_["fn"]):.4f}  Specificity={cc_["tn"] / (cc_["tn"] + cc_["fp"]):.4f}  '
      f'Precision={cc_["tp"] / (cc_["tp"] + cc_["fp"]):.4f}  NPV={cc_["tn"] / (cc_["tn"] + cc_["fn"]):.4f}')

## 8. Сохранение: `report.json`, `index.json`, модель

Фронт читает `artifacts/classification/flight_delays/report.json`. NaN/Inf заменяются на `null` (иначе `JSON.parse` в браузере падает).
Текст вывода собирается из реально посчитанных чисел — ничего не вписано вручную.

In [ ]:
def pct_(x, d=1):
    return f'{x * 100:.{d}f}%'


best_baseline = min((e for e in entries if e['is_baseline']), key=lambda e: e['cost_per_1000'])
saving = 1 - best['cost_per_1000'] / best_baseline['cost_per_1000']
cc = best['confusion_counts']
best_auc_name = max(models_only, key=lambda e: e['roc_auc'])['name']
best_ap_name = max(models_only, key=lambda e: e['ap'])['name']
best_brier_name = min(models_only, key=lambda e: e['brier'])['name']
PREV_TEST = float(FOLDS['test']['prevalence'])      # метрики считаются на тесте → и ориентиры тоже по тесту
brier_ref_ = PREV_TEST * (1 - PREV_TEST)
theory_thr = COST_FP / (COST_FP + COST_FN)

SPARK_BLOCK = {**SPARK_INFO, 'n_rows_raw': int(N_RAW), 'n_rows_used': int(N_USED), 'timings': TIMINGS,
               'tree_train_rows': int(N_TRAIN * frac_tree), 'ui': {'master': ':8085', 'history': ':18080'}}

FEAT_TXT = (f'Прогноз строится за {PRED_HORIZON_H} ч до планового вылета: расписание, перевозчик, маршрут, загрузка аэропорта'
            + (', предыдущий рейс борта и недавние задержки в аэропортах' if USE_HISTORY else '') + '. ')
_runner = min(PAIRED.items(), key=lambda kv: kv[1]['diff_mean']) if PAIRED else None
_bc = best['cost_ci']
CI_TXT = (f'95% ДИ цены (bootstrap по {_bc["n_days"]} дням теста): {_bc["lo"]:,.0f}–{_bc["hi"]:,.0f}{COST_UNIT}. '
          + (f'Ближайший конкурент ({_runner[0]}) дороже лишь в {_runner[1]["p_best_cheaper"]:.0%} выборок'
             + (' — разница в пределах шума.' if _runner[1]['p_best_cheaper'] < 0.9 else ' — разница устойчива.') if _runner else ''))

narrative = {
    'context': (f'BTS 2015: {N_RAW:,} рейсов, после очистки {N_USED:,}. Задержкой считаем прибытие позже плана на ≥15 минут '
                f'({pct_(PREVALENCE)} рейсов всего, {pct_(PREV_TEST)} в тесте). {FEAT_TXT}'
                'Разбиение по календарным дням исключает утечку через общую погоду и загрузку.'),
    'hero_metric': f'Потери на 1000 рейсов, {COST_UNIT}',
    'why_hero': (f'FP стоит {COST_FP:.0f}{COST_UNIT}, FN — {COST_FN:.0f}{COST_UNIT}: ошибки несимметричны, поэтому сравниваем модели в деньгах, '
                 'а не в accuracy. Цены можно менять во фронте — оптимальный порог пересчитается.'),
    'why_not_accuracy': (f'Baseline «всегда вовремя» имеет accuracy {pct_(next(e for e in entries if e["name"].endswith("вовремя"))["metrics"]["accuracy"])} '
                         'и при этом не ловит ни одной задержки. Accuracy не отражает цену пропусков.'),
    'main_conclusion': (f'Лучшая модель — {best["name"]}: {best["cost_per_1000"]:,.0f}{COST_UNIT} на 1000 рейсов против '
                        f'{best_baseline["cost_per_1000"]:,.0f}{COST_UNIT} у лучшего baseline ({pct_(saving)} экономии). '
                        f'Порог на валидации: {best["threshold"]:.3f} (для идеально откалиброванной модели ≈ {theory_thr:.2f}). '
                        f'На тесте поймано {cc["tp"]:,} задержек, пропущено {cc["fn"]:,}, ложных тревог {cc["fp"]:,}. '
                        f'Лучшая по ROC-AUC: {best_auc_name}. {CI_TXT}'),
    'why_ap_brier': (f'AP считается только по положительному классу и не раздувается числом верно отпущенных рейсов; случайная модель даёт '
                     f'AP = prevalence теста = {pct_(PREV_TEST)}, лучшая ({best_ap_name}) — {max(e["ap"] for e in models_only):.3f}. '
                     f'Brier Score оценивает сами вероятности и не зависит от порога: ориентир prev·(1−prev) = {brier_ref_:.4f}, '
                     f'лучшая ({best_brier_name}) — {min(e["brier"] for e in models_only):.4f}. '
                     'Пороги по F0.5 (упор на precision) и F2 (упор на recall) подобраны на валидации отдельно для «задержки» и «вовремя».'),
    'rules': {
        'cost_first': 'Порог выбирается по цене ошибок на валидации, а не по умолчанию 0.5',
        'no_leakage': 'Только признаки, известные до вылета; разбиение по календарным дням',
        'horizon': f'Признаки истории — только то, что известно за {PRED_HORIZON_H} ч до планового вылета',
        'uncertainty': 'Разницу между моделями меньше доверительного интервала (bootstrap по дням) не считаем различием',
        'baselines': 'Любая модель сравнивается с «всегда вовремя» и «всегда задержка»',
        'ap_not_roc': 'На дисбалансе ранжирование оцениваем по AP, а не по ROC-AUC; вероятности — по Brier Score',
    },
}

DIAG = {
    'is_balanced': False,
    'prevalence_positive': float(PREVALENCE),
    'best_cost_per_1000': float(best['cost_per_1000']),
    'baseline_cost_per_1000': float(best_baseline['cost_per_1000']),
    'saving_vs_baseline': float(saving),
    'best_pr_auc': float(best['pr_auc']),
    'best_roc_auc': float(best['roc_auc']),
    'best_mcc': float(best['metrics']['mcc']),
    'best_ap': float(best['ap']),
    'best_brier': float(best['brier']),
    'best_brier_skill': None if best['metrics']['brier_skill'] is None else float(best['metrics']['brier_skill']),
    'tp': cc['tp'], 'fp': cc['fp'], 'fn': cc['fn'], 'tn': cc['tn'],
    'cost_optimal_threshold_theory': theory_thr,
    'val_threshold': float(best['threshold']),
    'prevalence_test': PREV_TEST,
    'best_cost_ci': best['cost_ci'],
    'paired_vs_best': PAIRED,
}

report = {
    'meta': {
        'task': TASK,
        'title': 'Flight Delays · цена ошибки',
        'subtitle': 'Порог выбирается по деньгам: FN дороже FP — оптимум смещается от 0.5',
        'dataset': 'usdot/flight-delays (Kaggle · BTS 2015)',
        'target': 'ARRIVAL_DELAY ≥ 15 мин',
        'classes': ['on_time', 'delayed'], 'positive_class': 'delayed',
        'features': TREE_FEATURES, 'seed': SEED, 'generated_at': datetime.now().isoformat(),
        'n_samples_raw': int(N_RAW), 'n_samples': int(N_USED), 'n_features': len(TREE_FEATURES),
        'n_train': int(N_TRAIN), 'n_val': int(N_VAL), 'n_test': int(N_TEST),
        'horizon_hours': PRED_HORIZON_H, 'use_history': bool(USE_HISTORY),
        'target_stats': {
            # фронт сравнивает метрики ТЕСТА с этим числом (ориентир AP, Brier p(1−p), lift) → кладём prevalence теста;
            # prevalence по всей выборке — в prevalence_overall
            'prevalence': PREV_TEST,
            'prevalence_overall': float(PREVALENCE),
            'prevalence_by_fold': {f: v['prevalence'] for f, v in FOLDS.items()},
            'imbalance_ratio': float((1 - PREVALENCE) / PREVALENCE),
        },
        'costs': {'fp': COST_FP, 'fn': COST_FN, 'unit': COST_UNIT},
        'split_note': 'Разбиение по календарным дням: hash(month, day) % 10 → 0–6 train, 7 val, 8–9 test',
        'spark': SPARK_BLOCK,
        'narrative': narrative,
    },
    'headline': {
        'best_model': best['name'], 'best_metric': 'cost_per_1000', 'best_metric_value': float(best['cost_per_1000']),
        'verdict': (f'{best["name"]} · {best["cost_per_1000"]:,.0f}{COST_UNIT}/1000 · экономия {pct_(saving)} · '
                    f'sens={pct_(best["metrics"]["sensitivity"])} · spec={pct_(best["metrics"]["specificity"])} · '
                    f'prec={pct_(best["metrics"]["precision_pos"])} · AP={best["ap"]:.3f} · Brier={best["brier"]:.4f}'),
        'diagnostic_verdict': DIAG,
    },
    'models': sorted(entries, key=lambda e: (e['is_baseline'], e['cost_per_1000'])),
    'bootstrap': BOOT_REPORT,
    'eda': {'by_hour': eda_hour, 'by_month': eda_month, 'by_dow': eda_dow, 'by_airline': eda_airline},
    'charts': {k.replace('.png', ''): k for k in ('eda.png', 'cost_and_pr.png', 'ap_brier.png', 'fbeta_threshold.png', 'confusion_matrix.png') if (ART_DIR / k).exists()},
}

# --- модель и метаданные ---
slug = best['name'].lower().replace(' ', '_')
spark_model_path = MODEL_DIR / f'flight_delays_{slug}_spark'
try:
    fitted[best['name']].write().overwrite().save(spark_model_path.as_uri())
    saved_model = str(spark_model_path.relative_to(PROJECT))
except Exception as e:
    print('⚠ модель Spark не сохранена:', type(e).__name__, str(e)[:200])
    saved_model = None

meta_path = MODEL_DIR / f'flight_delays_{slug}_{datetime.now():%Y%m%d_%H%M}.json'
dump_json({'task': TASK, 'model': best['name'], 'features': TREE_FEATURES, 'threshold': best['threshold'],
           'costs': report['meta']['costs'], 'metrics': best['metrics'], 'saved_at': datetime.now().isoformat(),
           'spark_model': saved_model}, meta_path)

report['artifacts'] = {'model_spark': saved_model, 'model_meta': str(meta_path.relative_to(PROJECT)),
                       'metrics_csv': str((ART_DIR / 'metrics_all_models.csv').relative_to(PROJECT))}
report_path = ART_DIR / 'report.json'
dump_json(report, report_path)


def upsert_index(entry):
    p = PROJECT / 'artifacts' / 'index.json'
    idx = json.loads(p.read_text()) if p.exists() else {'generated_at': None, 'tasks': []}
    idx['tasks'] = sorted([t for t in idx['tasks'] if t['id'] != entry['id']] + [entry], key=lambda t: t['id'])
    idx['generated_at'] = datetime.now().isoformat()
    dump_json(idx, p)


upsert_index({'id': TASK, 'title': report['meta']['title'], 'best_model': best['name'],
              'hero_metric': 'cost_per_1000', 'hero_value': float(best['cost_per_1000']), 'unit': COST_UNIT,
              'report': str(report_path.relative_to(PROJECT)), 'charts_dir': str(ART_DIR.relative_to(PROJECT))})

print('report →', report_path, f'({report_path.stat().st_size / 1024:.0f} КБ)')
print(report['headline']['verdict'])
print('время этапов:', json.dumps(TIMINGS, ensure_ascii=False))

## 9. Итог

* Данные: реальные рейсы США 2015, обработка и обучение — на Spark; драйвер получает только гистограммы и сводки.
* Метрики карточки: Sensitivity, Specificity, Precision, NPV, MCC, AP, Brier Score, F0.5 и F2.0 (`models[].metrics`); AP и Brier лежат также в `models[].ap` / `models[].brier`.
* Для каждой модели в `report.json` сохраняются: `calibration` (диаграмма калибровки), `val_best` (пороги по F0.5/F2 для «задержки» и «вовремя», подобранные на валидации) и `at_f_thresholds` (метрики на тесте при этих порогах).
* Метрика выбирается **структурой цены ошибок**, а не привычкой: при FN ≫ FP оптимальный порог уходит заметно ниже 0.5.
* Порог выбран на валидации, цена посчитана на тесте — иначе экономия была бы оптимистичной.
* Во фронте (`Классификация → Flight Delays`) можно менять цены FP/FN и двигать порог — счётчики берутся из сетки порогов в `report.json`.

**Артефакты:** `artifacts/classification/flight_delays/{report.json, eda.png, cost_and_pr.png, ap_brier.png, fbeta_threshold.png, confusion_matrix.png, metrics_all_models.csv}`, `models/classification/flight_delays/…`.

Остановить Spark и освободить ресурсы: `spark.stop()`.
* v2: добавлены признаки истории с горизонтом `PRED_HORIZON_H`, out-of-fold target-encoding, early stopping у GBT и **доверительные интервалы цены** (`models[].cost_ci`, `bootstrap`). Критерий успеха — нижняя граница ДИ цены новой модели ниже верхней границы ДИ v1 (≈15.4–15.9 тыс. $) и заметный рост AP относительно 0.294.